In [3]:
import numpy as np
from scipy.optimize import linear_sum_assignment #Phan cong tuyen tinh
def khatri_rao(A, B):
    # A, B phai cung so luong column
    R = A.shape[1]
    res = np.zeros((A.shape[0] * B.shape[0], R))
    for r in range(R):
        res[:, r] = np.kron(A[:, r], B[:, r])
    return res


def unfold(X, mode):
    # X_(1) = A(C (.) B)^T
    # X_(2) = B(C (.) A)^T
    # X_(3) = C(B (.) A)^T
    if mode == 1:
        return np.transpose(X, (0, 2, 1)).reshape(X.shape[0], -1)

    elif mode == 2:
        return np.transpose(X, (1, 2, 0)).reshape(X.shape[1], -1)

    elif mode == 3:
        return np.transpose(X, (2, 1, 0)).reshape(X.shape[2], -1)


def reconstruct(A, B, C):
    return np.einsum('ir,jr,kr->ijk', A, B, C)


np.random.seed(42)

# Size cua tensor
I, J, K = 3,3,3

R_true = 2

A_true = np.random.rand(I, R_true)
B_true = np.random.rand(J, R_true)
C_true = np.random.rand(K, R_true)

X_true = reconstruct(A_true, B_true, C_true)

X = X_true.copy()

R = R_true
L_max = 100

A = np.random.rand(I, R)
B = np.random.rand(J, R)
C = np.random.rand(K, R)

X_1 = unfold(X, 1)
X_2 = unfold(X, 2)
X_3 = unfold(X, 3)


for l in range(L_max):

    F_A = khatri_rao(C, B).T
    A = np.linalg.lstsq(F_A.T, X_1.T, rcond=None)[0].T

    F_B = khatri_rao(C, A).T
    B = np.linalg.lstsq(F_B.T, X_2.T, rcond=None)[0].T

    F_C = khatri_rao(B, A).T
    C = np.linalg.lstsq(F_C.T, X_3.T, rcond=None)[0].T


X_rec = reconstruct(A, B, C)

print("\nX_true\n")
print(X_true)


print("\nX_rec\n")
print(X_rec)

norm_X_true = np.linalg.norm(X_true)

error1 = (np.linalg.norm(X_rec - X_true)**2 / (norm_X_true**2))

print(f"Error trên X = {error1:.2e}")


print("\nA\n")
print(A)

print("\nB\n")
print(B)

print("\nC\n")
print(C)

def align(A_est, A_true):
    R = A_est.shape[1]
    cost_matran = np.zeros((R,R))
    scale_matran = np.zeros((R,R))
    for i in range(R):
        for j in range(R):
            col_est = A_est[:,i]
            col_true = A_true[:,j]

            s = np.dot(col_est, col_true) / np.dot(col_est, col_est)
            scale_matran[i,j] = s
            err = np.linalg.norm(s*col_est - col_true)**2
            cost_matran[i,j] = err
    row_id, col_id = linear_sum_assignment(cost_matran)
    A_aligned = np.zeros_like(A_est)
    for idx in range(R):
        i = row_id[idx]
        j = col_id[idx]
        s = scale_matran[i,j]
        A_aligned[:,j] = s*A_est[:,i]
    norm_A_true = np.linalg.norm(A_true)
    real_err = (np.linalg.norm(A_aligned - A_true)**2) / (norm_A_true**2)
    return A_aligned, real_err

A_aligned, real_err_A = align(A, A_true)
print("\nA_true\n",A_true)
print("\nA_aligned\n",A_aligned)
print(f"Error tren factor A = {real_err_A:.2e}")



X_true

[[[0.19296779 0.15498659 0.43874828]
  [0.33035886 0.16439966 0.42175038]
  [0.20221732 0.17052043 0.48622729]]

 [[0.14549995 0.10283388 0.28504456]
  [0.45629441 0.15774928 0.35631133]
  [0.13583661 0.10923256 0.30928131]]

 [[0.03623469 0.02642911 0.07366151]
  [0.10152477 0.03731051 0.08649561]
  [0.03480048 0.02833316 0.08037307]]]

X_rec

[[[0.19296012 0.15498798 0.43875548]
  [0.33036861 0.1643973  0.4217393 ]
  [0.20220744 0.17052227 0.48623672]]

 [[0.14550874 0.10283248 0.28503696]
  [0.45628885 0.15775055 0.35631743]
  [0.13584747 0.1092308  0.30927181]]

 [[0.03623603 0.0264289  0.07366037]
  [0.10152414 0.03731065 0.08649628]
  [0.03480212 0.02833291 0.08037167]]]
Error trên X = 6.31e-10

A

[[0.41046904 0.31645704]
 [0.80203851 0.19925336]
 [0.17095201 0.05192137]]

B

[[0.06472293 0.64580108]
 [0.66916593 0.52784115]
 [0.02299045 0.7231491 ]]

C

[[0.68254073 0.85545178]
 [0.14911155 0.73899218]
 [0.24954945 2.11444618]]

A_true
 [[0.37454012 0.95071431]
 [0.731